# Tutorial: Training and Deploying a Streaming Fraud-Detection Pipeline on Google Cloud

This tutorial walks through a single end-to-end portfolio project in **two parts**:

- **Part 1 — Train** ([tutorial_2.ipynb](tutorial_2.ipynb)). Provision a private cloud environment that builds a Docker image, trains a PySpark fraud-detection model on Google Cloud, and pushes a versioned scoring image to a private container registry.
- **Part 2 — Stream** ([tutorial_3.ipynb](tutorial_3.ipynb)). Provision a second, separate environment that pulls that scoring image, replays credit-card transactions through Google Cloud Pub/Sub, scores them in near-real time, and writes a fraud report.

This notebook ([tutorial_1.ipynb](tutorial_1.ipynb)) covers the shared setup: the problem, the dataset, the overall architecture, prerequisites, the end-to-end lifecycle, and the persistent **data stack** that both parts depend on. Read it first, then open Part 1 and Part 2.

> **A note on style.** The repository's source files (`*.tf`, `*.py`, `Dockerfile`, `startup.sh`) are commented inline for readability, they explain *how* each piece works. This notebook explains *why* each piece exists and how the pieces fit together. Where a snippet is shown inline below, it is the headline; the full file is linked.

## Contents

1. [Introduction](#10-introduction)
    - [1.1 The problem](#11-the-problem)
    - [1.2 Dataset](#12-dataset)
    - [1.3 Architecture](#13-architecture)
    - [1.4 Prerequisites](#14-prerequisites)
    - [1.5 End-to-end lifecycle](#15-end-to-end-lifecycle)
    - [1.6 Real world applications](#16-real-world-applications)
2. [Data Stack](#20-data-stack)
    - [2.1 Data Stack architecture](#21-data-stack-architecture)
    - [2.2 GCS bucket](#22-gcs-bucket)
    - [2.3 Artifact Registry](#23-artifact-registry)
    - [2.4 Service account and IAM](#24-service-account-and-iam)
3. [References](#references)

## 1.0 Introduction

### 1.1 The problem

Credit-card issuers process millions of transactions per day and have to decide, in milliseconds, whether each one is fraudulent. Doing that well requires two very different pieces of infrastructure: an offline **training** environment that periodically re-learns fraud patterns from historical labelled data, and an always-on **inference** environment that scores every live transaction as it arrives. This tutorial deploys both, a short-lived training stack that produces a versioned model, and a separate streaming stack that consumes that model and flags suspicious transactions in simulated near real time.


This tutorial covers two things: **training** an ML model in the cloud, and **deploying** it to score live transactions and flag suspected fraud. The two phases need different infrastructure — training is a short, heavy batch job; inference is a long-running, lightweight service — and the rest of the notebook builds the cloud architecture that supports both, then tears it down cleanly.


### 1.2 Dataset

The tutorial uses the **Credit Card Fraud Detection** dataset published by the Machine Learning Group at Université Libre de Bruxelles and hosted on [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud). It contains **284,807 real transactions** from European cardholders over two days in September 2013, with a binary `Class` label (0 = genuine, 1 = fraud).

Each row has 30 numeric features: a `Time` column, an `Amount` column, and 28 PCA-transformed components (`V1`–`V28`) that hide the original cardholder fields for confidentiality. Only **492 transactions (≈0.17%) are fraudulent**, which mirrors the real production challenge and is why the training script up-weights the positive class and reports AUC-PR alongside AUC-ROC.

### 1.3 Architecture

The project architecture is divided into three independent Terraform stacks. A stack is a folder of .tf files that Terraform applies or destroys as a single unit. Each stack corresponds to a distinct phase of the workload:

- **Data stack** — persistent shared resources (GCS bucket, Artifact Registry repo, service account) that outlive individual runs.
- **Train stack** — a short-lived VM that trains the model and pushes a versioned scoring image, then is torn down.
- **Stream stack** — a short-lived VM that pulls the scoring image and serves predictions over Pub/Sub, then is torn down.

<p align="center">
  <img src="images/architecture.png" width="600"><br>
  <em>Fig 1. Project architecture</em>
</p>

### 1.4 Prerequisites

- A GCP project with billing enabled and your `gcloud` CLI authenticated:
  ```bash
  gcloud auth login
  gcloud auth application-default login
  gcloud config set project <project_id>
  ```

- Terraform `>= 1.5` on `PATH`.

- The Kaggle [creditcard.csv](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) dataset placed at `data/creditcard.csv`. The file is gitignored; Terraform uploads it to GCS during Part 1.

- Container Analysis enabled once per project:
  ```bash
  gcloud services enable containerscanning.googleapis.com --project <project_id>
  ```

- Each stack has a gitignored `terraform.tfvars` that you need to create from the committed `terraform.tfvars.example` sitting next to it:
  ```bash
  cp infra/terraform/data/terraform.tfvars.example   infra/terraform/data/terraform.tfvars
  cp infra/terraform/train/terraform.tfvars.example  infra/terraform/train/terraform.tfvars
  cp infra/terraform/stream/terraform.tfvars.example infra/terraform/stream/terraform.tfvars
  ```
  Only the **data** stack's file needs editing: set `project_id` to your GCP project and optionally change `region`. Train and stream defaults (zone, machine type, VPC, CIDR ranges) work as-is.

### 1.5 End-to-end lifecycle

From a clean project, six commands take you all the way through and back:

```bash
1. cd infra/terraform/data   && terraform init && terraform apply   # bucket + registry + SA
2. cd ../train               && terraform init && terraform apply   # train VM trains model + pushes scoring image
3. cd ../train               && terraform destroy                    # train compute gone, registry image stays
4. cd ../stream              && terraform init && terraform apply   # stream VM pulls image, scores Pub/Sub, writes report
5. cd ../stream              && terraform destroy                    # stream compute gone
6. cd ../data                && terraform destroy                    # bucket + registry gone — zero cloud state
```

Or run [scripts/destroy_everything.sh](scripts/destroy_everything.sh) (Linux/Mac) or [scripts/destroy_everything.ps1](scripts/destroy_everything.ps1) (Windows) at the end to do steps 3+5+6 in one shot.

### 1.6 Real world applications

The training stack fits any workflow that ships a model on a schedule as a versioned artefact, for example fraud, churn, or credit risk scoring.

The streaming stack fits any low latency decisioning flow where events must be scored in near real time, for example card swipes, IoT telemetry, or content moderation.

Combined, they form a full train once and serve many lifecycle where a batch job publishes the model to the registry and a long running consumer scores live events against it.

## 2.0 Data Stack
The train and stream VMs are temporary: they only need to exist while a training or streaming run is happening. But the dataset, trained model, scoring image, and the shared service account all need to remain available across multiple runs.

To achieve this, the project uses an independent data stack for persistent resources. It is created before the train and stream stacks so the dataset, registry, and shared service account are all in place, and destroyed last so they remain available across repeated runs.


### 2.1 Data Stack architecture

<p align="center">
  <img src="images/data_stack.png" width="600"><br>
  <em>Fig 2. Data Stack architecture</em>
</p>

As shown in the figure above, the service account sits at the centre of the data stack as a shared identity. The train stack uses it to read the dataset from the bucket and push the trained image to the registry, while the stream stack uses it to pull that image and write the fraud report back to the bucket. Pub/Sub is dashed because the topic is created by the stream stack; the data stack only grants publisher/subscriber permissions so the SA is ready when the topic appears.

### 2.2 GCS bucket

The bucket is the only persistent storage the project uses. Terraform pre-loads the full training dataset (`creditcard.csv`), and a subset used to simulate live credit card transactions for the stream stack (`transactions_stream.csv`). At runtime the bucket also receives the outputs the VMs produce (`summary.json` from training and `fraud_report.json` from streaming). Because each object is independent, both VMs can read and write the bucket concurrently.

### 2.3 Artifact Registry

A private Artifact Registry repo is used instead of GCS because Docker can push and pull from it directly, and shared image layers are reused rather than stored repeatedly. It starts empty and is filled by the train VM, which pushes two images: `fraud-detection:base` (environment + training code) and `fraud-scoring:v5` (base plus the trained model and streaming entrypoint). The base image rarely changes; the scoring image is updated between training runs.


### 2.4 Service account and IAM

Both VMs run as a single shared service account, `fraud_vm_sa`. One SA is used for both stacks because they need almost identical permissions, and consolidating keeps IAM auditable.

Each role is granted at the tightest scope possible. The service account gets read and write on the dataset bucket only, push and pull on the registry only, and publish and subscribe on Pub/Sub across the whole project. Pub/Sub has to be project-wide because the topic itself does not exist yet; it is created later by the stream stack.

The SA itself and its bucket binding look like this; the registry and Pub/Sub bindings follow the same `_iam_member` pattern against the relevant resource:

```hcl
# infra/terraform/data/main.tf (abridged)
resource "google_service_account" "fraud_vm_sa" {
  account_id   = var.vm_service_account_id
  display_name = "Fraud Detection VM (shared, used by train and stream)"
}

resource "google_storage_bucket_iam_member" "dataset_admin" {
  bucket = google_storage_bucket.dataset.name
  role   = "roles/storage.objectAdmin"
  member = "serviceAccount:${google_service_account.fraud_vm_sa.email}"
}
```

> **Deploy now.** To stand the data stack up as you read, run `terraform init && terraform apply` in `infra/terraform/data/`. It provisions in under a minute and leaves the bucket and registry in place for Parts 1 and 2.

## References

- Machine Learning Group, Université Libre de Bruxelles. *Credit Card Fraud Detection.* Kaggle. https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud